# Revision protocol: largest data regime, one task

Runs the budget-matched comparison (SFT, continued SFT, SFT->DPO with four negative strategies, SFT->KTO)
for one task at its largest train size, then prints the paired statistics.

Run all cells once. Finished stages are skipped, so the notebook can be re-run after a disconnect.

In [ ]:
TASK_KEY = 'ds4_task2'            # largest regime available: 1000 training examples
TRAIN_SIZE = 1000
SEEDS = [11, 22, 33]
LEARNING_RATES = [5e-6, 2e-5, 1e-4]
SKIP_BASE_START = True            # skip DPO/KTO without prior SFT
BASE_CONFIG = 'llama31_revision.yaml'
USE_DRIVE = True                  # keep outputs on Google Drive so a disconnect loses nothing
MIN_GPU_GB = 20                   # stop early on a small GPU such as the 15 GB T4; set to 0 to run anyway

## 1. Get the code

Clones the public repository, then overlays the revision code that is not pushed yet.

In [ ]:
import base64, io, os, tarfile
from pathlib import Path

REPO_DIR = Path('/content/Beyond-SFT')
if not REPO_DIR.exists():
    !git clone --depth 1 https://github.com/HydnumRepandum/Beyond-SFT.git {REPO_DIR}

# Uncommitted revision files (.gitignore, scripts/train.py, src/topic_alignment/config.py, src/topic_alignment/data.py, src/topic_alignment/training.py, configs/gemma3_revision.yaml, configs/llama31_revision.yaml, docs/REVISION_PROTOCOL.md, scripts/score_generation.py, scripts/stats.py, src/topic_alignment/generation.py, src/topic_alignment/preferences.py, src/topic_alignment/stats.py, tests/test_revision_protocol.py).
# Remove this cell once the revision branch is pushed, and clone that branch instead.
REVISION_PAYLOAD = (
    '...'
    'B4M1AKgCAA=='
)
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(REVISION_PAYLOAD)), mode='r:gz') as archive:
    archive.extractall(REPO_DIR)
os.chdir(REPO_DIR)
print('revision files:', sorted(tarfile.open(fileobj=io.BytesIO(base64.b64decode(REVISION_PAYLOAD))).getnames()))

## 2. Install dependencies and check the GPU

No runtime restart is needed: every training stage runs in its own fresh Python process.

In [ ]:
import subprocess
from pathlib import Path

setup_flag = Path('/content/.revision_setup_done')
if not setup_flag.exists():
    subprocess.run(['bash', 'scripts/colab_setup.sh'], check=True)
    setup_flag.write_text('done')
print('Dependencies installed.')

gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader,nounits'],
                     capture_output=True, text=True).stdout.strip()
print('GPU:', gpu or 'none')
gpu_gb = float(gpu.split(',')[-1]) / 1024 if gpu else 0.0
if gpu_gb < MIN_GPU_GB:
    raise RuntimeError(
        f'This runtime has {gpu_gb:.0f} GB of GPU memory ({gpu or "no GPU"}). The grid was sized for a large GPU '
        '(the paper used a 96 GB G4). Connect to a G4 or A100 runtime, or set MIN_GPU_GB = 0 to run anyway.'
    )

## 3. Output location and runtime config

In [ ]:
import yaml
from pathlib import Path

PERSIST_ROOT = Path('/content/revision_outputs')
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        PERSIST_ROOT = Path('/content/drive/MyDrive/beyond_sft_alignment/revision')
    except Exception as error:
        print('Drive not mounted, outputs stay on the runtime disk:', error)
PERSIST_ROOT.mkdir(parents=True, exist_ok=True)

model_slug = BASE_CONFIG.split('_')[0]
runtime_config = {
    'extends': BASE_CONFIG,
    'project': {
        'outputs_root': str(PERSIST_ROOT / 'outputs' / model_slug),
        'results_root': str(PERSIST_ROOT / 'results' / model_slug),
    },
    'tracking': {'enabled': False},
}
CONFIG_PATH = REPO_DIR / 'configs' / '_runtime_revision.yaml'
CONFIG_PATH.write_text(yaml.safe_dump(runtime_config, sort_keys=False))
print('config:', CONFIG_PATH)
print('outputs:', PERSIST_ROOT)

## 4. Run the grid

Per seed: SFT, out-of-fold SFT scoring, then for each learning rate the continued-SFT control, four SFT->DPO variants and SFT->KTO.

In [ ]:
seeds = ' '.join(str(seed) for seed in SEEDS)
rates = ' '.join(str(rate) for rate in LEARNING_RATES)
skip = '--skip-base-start' if SKIP_BASE_START else ''
command = f'python scripts/train.py protocol --config {CONFIG_PATH} --task {TASK_KEY} --seeds {seeds} --train-sizes {TRAIN_SIZE} --learning-rates {rates} {skip}'
print(command)
get_ipython().system(command)   # same as !command, with live output
if _exit_code != 0:
    raise RuntimeError('The grid stopped with an error. The cause is in the last lines printed above this message.')

## 5. Results

`difference` is treatment minus reference, in proportion points (0.01 = 1 pp), with a 95% bootstrap interval.
The budget-matched reference is `sft_sft` (continued SFT at the same learning rate and steps).

In [ ]:
import pandas as pd

get_ipython().system(f'python scripts/stats.py --config {CONFIG_PATH} --task {TASK_KEY}')
tables = PERSIST_ROOT / 'results' / model_slug / TASK_KEY / 'tables'
pd.set_option('display.max_rows', 200, 'display.width', 250, 'display.max_columns', 40)
columns = ['method', 'preference_strategy', 'learning_rate', 'seed_count',
           'accuracy_difference', 'accuracy_ci_low', 'accuracy_ci_high', 'accuracy_p_holm',
           'macro_f1_difference', 'macro_f1_ci_low', 'macro_f1_ci_high', 'macro_f1_p_holm',
           'items_fixed_mean', 'items_broken_mean']
for name in ['paired_contrasts_vs_sft_sft.csv', 'paired_contrasts_vs_sft.csv']:
    path = tables / name
    if path.exists():
        print('\n' + name)
        print(pd.read_csv(path)[columns].round(4).to_string(index=False))
summary_path = tables / 'summary_by_configuration.csv'
if summary_path.exists():
    print('\nsummary_by_configuration.csv')
    print(pd.read_csv(summary_path).round(4).to_string(index=False))
else:
    print('No finished runs yet, so there are no statistics to show.')